In [1]:
import dask
from dask.distributed import Client

In [2]:
client = Client(n_workers=4, threads_per_worker=1, memory_limit='2GB')
client

/opt/anaconda3/lib/python3.12/site-packages/distributed/node.py:182: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 54650 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:54650/status,
Dashboard: http://127.0.0.1:54650/status,Workers: 4
Total threads: 4,Total memory: 7.45 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:54651,Workers: 4
Dashboard: http://127.0.0.1:54650/status,Total threads: 4
Started: Just now,Total memory: 7.45 GiB
Comm: tcp://127.0.0.1:54665,Total threads: 1
Dashboard: http://127.0.0.1:54668/status,Memory: 1.86 GiB
Nanny: tcp://127.0.0.1:54654,


In [3]:
import dask
df = dask.datasets.timeseries()

/opt/anaconda3/lib/python3.12/site-packages/dask_expr/_collection.py:5833: UserWarning: dask_expr does not support the DataFrameIOFunction protocol for column projection. To enable column projection, please ensure that the signature of `func` includes a `columns=` keyword argument instead.
  warnings.warn(


In [4]:
df

,name,id,x,y
npartitions=30,,,,
2000-01-01,string,int64,float64,float64
2000-01-02,...,...,...,...
...,...,...,...,...
2000-01-30,...,...,...,...
2000-01-31,...,...,...,...


In [5]:
df.compute().head(5)

2026-02-20 00:46:02,133 - distributed.worker - ERROR - failed during get data with tcp://127.0.0.1:54662 -> tcp://127.0.0.1:54665
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.12/site-packages/tornado/iostream.py", line 962, in _handle_write
    num_bytes = self.write_to_fd(self._write_buffer.peek(size))
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.12/site-packages/tornado/iostream.py", line 1124, in write_to_fd
    return self.socket.send(data)  # type: ignore
           ^^^^^^^^^^^^^^^^^^^^^^
OSError: [Errno 55] No buffer space available

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.12/site-packages/distributed/worker.py", line 1783, in get_data
    response = await comm.read(deserializers=serializers)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.12/site-packages/dis

,name,id,x,y
timestamp,,,,
2000-01-01 00:00:00,Alice,894,0.386141,0.508339
2000-01-01 00:00:01,Patricia,956,0.577245,0.246441
2000-01-01 00:00:02,Oliver,999,-0.413937,-0.431332
2000-01-01 00:00:03,Quinn,1057,-0.271495,-0.728492
2000-01-01 00:00:04,Tim,969,-0.837131,-0.451577


Use Standard Pandas Operations.

In [7]:
df2 = df[df.y > 0]
df2

,name,id,x,y
npartitions=30,,,,
2000-01-01,string,int64,float64,float64
2000-01-02,...,...,...,...
...,...,...,...,...
2000-01-30,...,...,...,...
2000-01-31,...,...,...,...


In [8]:
df3 = df2.groupby("name").x.std()
df3

Dask Series Structure:
npartitions=1
    float64
        ...
Dask Name: getitem, 8 expressions
Expr=(((Filter(frame=ArrowStringConversion(frame=FromMap(8ef37c1)), predicate=ArrowStringConversion(frame=FromMap(8ef37c1))['y'] > 0))[['name', 'x']]).std(ddof=1, numeric_only=False, split_out=None, observed=False))['x']

In [9]:
df3.compute()

2026-02-20 00:46:02,830 - distributed.protocol.core - CRITICAL - Failed to deserialize
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.12/site-packages/msgpack/fallback.py", line 128, in unpackb
    ret = unpacker._unpack()
          ^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.12/site-packages/msgpack/fallback.py", line 565, in _unpack
    ret.append(self._unpack(EX_CONSTRUCT))
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.12/site-packages/msgpack/fallback.py", line 585, in _unpack
    key = self._unpack(EX_CONSTRUCT)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.12/site-packages/msgpack/fallback.py", line 546, in _unpack
    typ, n, obj = self._read_header()
                  ^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.12/site-packages/msgpack/fallback.py", line 447, in _read_header
    self._reserve(1)
  File "/opt/anaconda3/lib/python3.12/site-packages/msgpack/fallback.py", line 420, 

CancelledError: ('repartitiontofewer-91bdf7eed289349cc448cc6908fceff3', 0)

In [ ]:
df4 = df.groupby("name").aggregate({"x": "sum", "y": "max"})
df4

In [ ]:
df4.compute().head(5)

In [ ]:
#在这里讲的dashboard

In [30]:
# benchmark
import dask
from dask.distributed import performance_report

with performance_report(filename="dask_report.html"):
    df = dask.datasets.timeseries(start='2000-01-01', end='2004-01-01')
    df.groupby(df.name).x.mean().compute()

/opt/anaconda3/lib/python3.12/site-packages/dask_expr/_collection.py:5833: UserWarning: dask_expr does not support the DataFrameIOFunction protocol for column projection. To enable column projection, please ensure that the signature of `func` includes a `columns=` keyword argument instead.
  warnings.warn(
2026-02-20 05:10:19,467 - distributed.protocol.core - CRITICAL - Failed to deserialize
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.12/site-packages/distributed/protocol/core.py", line 175, in loads
    return msgpack.loads(
           ^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.12/site-packages/msgpack/fallback.py", line 136, in unpackb
    File-like object having `.read(n)` method.
msgpack.exceptions.ExtraData: unpack(b) received extra data.
2026-02-20 05:10:19,469 - distributed.core - ERROR - Exception while handling op register-client
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.12/site-packages/distributed/core.py", line 97

CancelledError: ('repartitiontofewer-d1357858d5815558e764053b49517945', 0)